## CarpeDIAMS

A package for deconvoluting dia/aif/msn data into pseudo-dda data. Builds on the foundation of corrdec and ms-dial but integrates new clustering methods of ms2 data and more sophisticated filtering methods for the deconvolution.

It is all about catching the DIA patterns in our MS data.

Three types of input can be used to generate an mgf of ms2 data:
* An MsExperiment object from xcms
* A mzml directory path and a target list of peak positions (mz and rt)
* mzml file(s) which are submitted to peak picking and then processed by the carpediams workflow


### How it works

As with corrdec and msdial the algorithm simply finds ms2 fragments that correlate with 

In [1]:
library(Spectra)
library(xcms)

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, saveRDS, setdiff,
    table, tapply, union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: BiocParallel


This is xcms version 4.4.0 



Attaching package: ‘xcms’


The following ob

In [2]:
pak::pak("https://github.com/JohanLassen/carpeDIAMS")

! Using bundled GitHub PAT. Please add your own PAT using `gitcreds::gitcreds_set()`.


✔ Updated metadata database: 3.31 MB in 2 files.


ℹ Updating metadata database
✔ Updating metadata database ... done


 

→ Will install 1 package.

→ The package (0 B) is cached.

+ carpeDIAMS   0.1.0 [bld][cmp] (GitHub: 1f3350c) + ✔ python3

✖ Missing 7 system packages. You'll probably need to install them manually:

+ glpk-devel     - igraph     
+ libcurl-devel  - arrow, curl
+ libicu-devel   - stringi    
+ libpng-devel   - png        
+ libuv-devel    - fs         
+ libxml2-devel  - igraph, XML
+ netcdf-devel   - ncdf4      

ℹ No downloads are needed, 1 pkg is cached

✔ Got carpeDIAMS 0.1.0 (source) (1.26 MB)

ℹ Packaging carpeDIAMS 0.1.0

✔ Packaged carpeDIAMS 0.1.0 (2.3s)

ℹ Building carpeDIAMS 0.1.0

✔ Built carpeDIAMS 0.1.0 (9.4s)

✔ Installed carpeDIAMS 0.1.0 (github::JohanLassen/carpeDIAMS@1f3350c) (1.1s)

✔ 1 pkg + 141 deps: kept 130, added 1, dld 1 (NA B) [40.9s]



In [6]:
library(carpeDIAMS)
library(tidyr)
library(dplyr)
library(purrr)
library(tibble)
library(readr)


Attaching package: ‘tidyr’


The following object is masked from ‘package:S4Vectors’:

    expand



Attaching package: ‘dplyr’


The following objects are masked from ‘package:xcms’:

    collect, groups


The following objects are masked from ‘package:S4Vectors’:

    first, intersect, rename, setdiff, setequal, union


The following objects are masked from ‘package:BiocGenerics’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [12]:
files <- list.files("/faststorage/project/forensics_TOFscreenings/BACKUP/TOF2_stoffer_compassxport", recursive = TRUE, full.names = TRUE)

tmp_folder <- "/faststorage/project/forensics_TOFscreenings/06_integrated_workflow/results_all/drugs_annotation/tmp"

df <- tibble::tibble(
  file = files,
  sample_name = basename(files),
  file_size = file.size(files)
) |> 
mutate(
    peak_picked = paste0(tmp_folder, "/", gsub(".mzML$", "_peaks.rds", sample_name)),
    integration_ready = paste0(tmp_folder, "/", "integration_ready.rds"),
    integration1 = "#"
)

library(stringr)
TOF2_DB <- readr::read_csv("/faststorage/project/forensics_TOFscreenings/06_integrated_workflow/results_all/drugs_annotation/TOF-2_DB.csv")

COMPOUND_LIST <- readr::read_csv("/faststorage/project/forensics_TOFscreenings/06_integrated_workflow/results_all/drugs_annotation/compound_file_mapping.csv")

normalize <- function(x) {
  x |>
    tolower() |>
    str_replace_all("[\\-_\\. ]", "")
}

norm_db    <- normalize(TOF2_DB$name)
norm_files <- normalize(COMPOUND_LIST$basename)

# For each db name, find which filenames contain it
match_idx <- sapply(norm_db, function(db) {
  which(str_detect(norm_files, fixed(db)))
})

result <- data.frame(
  db_idx   = rep(seq_along(norm_db), lengths(match_idx)),
  file_idx = unlist(match_idx)
)

# Join in the actual columns
result <- data.frame(
  db_idx   = rep(seq_along(norm_db), lengths(match_idx)),
  file_idx = unlist(match_idx)
) |>
  mutate(
    TOF2_DB[db_idx, ],
    COMPOUND_LIST[file_idx, ]
  )

missing_frags <- result  |> 
  # mutate(
  #   `QI 1` = as.numeric(`QI 1`), 
  #   `QI 2` = as.numeric(`QI 2`), 
  #   `QI 3` = as.numeric(`QI 3`))  |> 
  select(`QI 1`, `QI 2`, `QI 3`) |> 
  map_dfc(is.na) |> 
  as.matrix() |> rowSums()

result <- result[missing_frags<3, ]



library(Spectra)
library(BiocParallel)
register(SerialParam())

mzml_path <- "/faststorage/project/forensics_TOFscreenings/BACKUP/TOF2_stoffer_compassxport/indkoering_forsoeg/indkoering_stoffer/gruppe_indkoering/"
origin_path <- "O:/HE_IFR-Rka/Instrumentdata/toks/UPLC-TOF/UPLC-TOF-2/indkoering_forsoeg/indkoering_stoffer/gruppe_indkoering/"
result_existing <- result |> mutate(mzml = gsub(origin_path, mzml_path, file), mzml = gsub(".d$", ".mzML", mzml))
result_existing <- result_existing |> filter(file.exists(mzml))

compounds <- result_existing$name |> unique()
files <- result_existing |> filter(name %in% compounds[1:5]) 


Rows: 951 Columns: 25
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (12): formula, name, CAS, QI 1, QI 2, QI 3, QI 1 min, comment3, comment4...
dbl (12): m/z, rt, comment 1, comment 2, minimum area, indivSigma, indivMass...
lgl  (1): relativeRetentiontime

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 2505 Columns: 3
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (3): compound, basename, file

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [ ]:
sps <- Spectra(files$mzml, backend = MsBackendMzR())

In [ ]:
library(MsExperiment); library(xcms); library(MsExperiment); library(Spectra)

cwp        <- CentWaveParam(
  peakwidth = c(2,30),
  snthresh  = 6,
  ppm       = 12,
  prefilter = c(3, 1000),
  mzdiff    = 0.01)

sps1 <- filterMsLevel(sps, 1L)
mse <- MsExperiment()
spectra(mse) <- sps1

sampleData(mse) <- DataFrame(sample_name = unique(dataOrigin(sps1)))   # MUST equal dataOrigin
mse <- linkSampleData(mse, with = "sampleData.sample_name = spectra.dataOrigin")
xdata <- findChromPeaks(mse, cwp)


[---------------------------------------------------------] 0/14 (  0%) in  0s

[===>-----------------------------------------------------] 1/14 (  7%) in  1m

[=======>-------------------------------------------------] 2/14 ( 14%) in  2m

[===========>---------------------------------------------] 3/14 ( 21%) in  3m

[===============>-----------------------------------------] 4/14 ( 29%) in  4m

[===================>-------------------------------------] 5/14 ( 36%) in  5m

[=======================>---------------------------------] 6/14 ( 43%) in  6m

[===========================>-----------------------------] 7/14 ( 50%) in  7m

[================================>------------------------] 8/14 ( 57%) in  8m

[====================================>--------------------] 9/14 ( 64%) in  9m

[=======================================>----------------] 10/14 ( 71%) in 10m

[===========================================>------------] 11/14 ( 79%) in 11m

[======================================

In [19]:
xdata

Object of class XcmsExperiment 
 Spectra: MS1 (157815) 
 Experiment data: 27 sample(s)
 Sample data links:
  - spectra: 27 sample(s) to 157815 element(s).
 xcms results:
  - chromatographic peaks: 52310 in MS level(s): 1 